# Parda — Phase 3a: OCR the TRAINING documents (CPU only, free)

Runs Tesseract on 10,000 documents (3,334 per language) so the Phase 3 PII model can learn
from real OCR mistakes. The 300 benchmark pages are **excluded** so the final test stays fair.

**Settings:** Accelerator **None** · Internet **On** · Input: your **Parda** (Phase 1) notebook output.
Check cells 1–4, then **Save Version → Save & Run All** (~1.5 h), close the tab.

In [ ]:
# 1. Get the latest code
import os, subprocess
from kaggle_secrets import UserSecretsClient
GH = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Tesseract + Hindi + Kannada (the EasyOCR install in this script is harmless, just unused here)
!bash scripts/setup_ocr.sh

In [ ]:
# 3. Dataset (should print 30000 / 30000)
from parda.ocr.run_ocr import find_data, _n_images
DATA = find_data()
print("DATA =", DATA, "| images:", _n_images(DATA))
OUT = "/kaggle/working/ocr_train"

In [ ]:
# 4. Tests: every line ok
!python tests/test_ocr.py

In [ ]:
# 5. OCR 10,000 training pages with Tesseract (4 CPU processes, ~1.5 h). Resumable.
!python -m parda.ocr.run_ocr --engine tesseract --data {DATA} --out {OUT} --per_lang 3334 --seed 42 --exclude_benchmark --workers 4
!wc -l {OUT}/ocr_tesseract.jsonl && du -sh {OUT}

## After the run
Nothing to download. In Phase 3 you'll attach **this notebook's output** as an Input (like you did for Phase 1).